In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import math

In [ ]:
img_path = "../../images/lenna.png"
img = cv.imread(img_path, 0)

cv.imshow("Original Image", img)
cv.waitKey(0)
cv.destroyAllWindows()

1. Noises (Gaussian, Uniform, Impulse, Speckle, Poisson, Rayleigh, Periodic)

Each noise model perturbs pixel intensities differently:
- Gaussian: adds values drawn from a normal distribution N(0, sigma^2).
- Uniform: adds values drawn evenly from a range [a, b].
- Impulse (salt-and-pepper): randomly forces some pixels to 0 (pepper) or 255 (salt).
- Speckle: multiplicative noise, f + f * n, common in radar/ultrasound images.
- Poisson: signal-dependent "shot noise", where brighter pixels get noisier.
- Rayleigh: values drawn from a Rayleigh distribution, skewed towards small positive values.
- Periodic: a repeating sinusoidal pattern added across the image, from interference sources.

In [ ]:
height, width = img.shape

# Gaussian noise
gauss_noise = np.random.normal(0, 20, img.shape)
gaussian_img = np.clip(img + gauss_noise, 0, 255).astype(np.uint8)

# Uniform noise
uniform_noise = np.random.uniform(-40, 40, img.shape)
uniform_img = np.clip(img + uniform_noise, 0, 255).astype(np.uint8)

# Impulse (salt and pepper) noise
prob = 0.05
rnd = np.random.rand(height, width)

impulse_img = img.copy()
impulse_img[rnd < prob / 2] = 0
impulse_img[rnd > 1 - prob / 2] = 255

# Speckle noise
speckle_noise = np.random.normal(0, 0.2, img.shape)
speckle_img = np.clip(img + img * speckle_noise, 0, 255).astype(np.uint8)

# Poisson noise
poisson_img = np.clip(np.random.poisson(img), 0, 255).astype(np.uint8)

# Rayleigh noise
rayleigh_noise = np.random.rayleigh(20, img.shape)
rayleigh_img = np.clip(img + rayleigh_noise, 0, 255).astype(np.uint8)

# Periodic noise
u = np.arange(width)
periodic_noise = 20 * np.sin(2 * np.pi * u / 10)
periodic_img = np.clip(img + periodic_noise, 0, 255).astype(np.uint8)

cv.imshow("Original", img)
cv.imshow("Gaussian Noise", gaussian_img)
cv.imshow("Uniform Noise", uniform_img)
cv.imshow("Impulse Noise", impulse_img)
cv.imshow("Speckle Noise", speckle_img)
cv.imshow("Poisson Noise", poisson_img)
cv.imshow("Rayleigh Noise", rayleigh_img)
cv.imshow("Periodic Noise", periodic_img)

cv.waitKey(0)
cv.destroyAllWindows()

2. Filters for Noise Minimization
a. Mean filter
b. Averaging and weighted averaging

Linear smoothing filters replace each pixel with a weighted average of its neighbourhood:
- Mean/box filter: every neighbour gets equal weight $1/n^2$.
- Weighted average: a kernel such as $\frac{1}{16}\begin{bmatrix}1&2&1\\2&4&2\\1&2&1\end{bmatrix}$
  gives the centre pixel more importance, smoothing noise while keeping a bit more detail than a
  plain mean filter.

In [ ]:
noisy = gaussian_img


def mean_filter(img, size):

    height, width = img.shape
    pad = size // 2

    padded = np.pad(img, pad, mode="edge")
    result = np.zeros_like(img)

    for i in range(height):
        for j in range(width):
            result[i, j] = np.mean(padded[i:i+size, j:j+size])

    return result


mean3 = mean_filter(noisy, 3)
average3 = cv.blur(noisy, (3, 3))

weighted_kernel = np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
]) / 16

weighted3 = cv.filter2D(noisy, -1, weighted_kernel)

cv.imshow("Noisy", noisy)
cv.imshow("Mean Filter", mean3)
cv.imshow("Averaging Filter", average3)
cv.imshow("Weighted Averaging Filter", weighted3)

cv.waitKey(0)
cv.destroyAllWindows()

3. Non-linear spatial domain filter
a. Alpha trimmed mean filter
b. Minimum filter
c. Maximum filter
d. Median filter

Order-statistic filters sort the pixel values in a neighbourhood and pick a statistic instead of
averaging:
- Median: the middle value, very effective against impulse noise since it ignores outliers.
- Minimum: darkens the image, good for removing salt (bright) noise, but erodes bright detail.
- Maximum: brightens the image, good for removing pepper (dark) noise, but erodes dark detail.
- Alpha-trimmed mean: sort the neighbourhood, discard the d/2 smallest and d/2 largest values,
  then average what remains - a compromise between the mean and the median filter.

In [ ]:
def order_filter(img, size, func):

    height, width = img.shape
    pad = size // 2

    padded = np.pad(img, pad, mode="edge")
    result = np.zeros_like(img)

    for i in range(height):
        for j in range(width):
            result[i, j] = func(padded[i:i+size, j:j+size])

    return result


def alpha_trimmed_mean_filter(img, size, d):

    height, width = img.shape
    pad = size // 2

    padded = np.pad(img, pad, mode="edge")
    result = np.zeros_like(img)

    for i in range(height):
        for j in range(width):
            vals = np.sort(padded[i:i+size, j:j+size].flatten())
            result[i, j] = np.mean(vals[d // 2: len(vals) - d // 2])

    return result


sp_noisy = impulse_img

alpha_trimmed = alpha_trimmed_mean_filter(sp_noisy, 3, 2)
minimum = order_filter(sp_noisy, 3, np.min)
maximum = order_filter(sp_noisy, 3, np.max)
median = order_filter(sp_noisy, 3, np.median)

cv.imshow("Noisy", sp_noisy)
cv.imshow("Alpha Trimmed Mean", alpha_trimmed)
cv.imshow("Minimum Filter", minimum)
cv.imshow("Maximum Filter", maximum)
cv.imshow("Median Filter", median)

cv.waitKey(0)
cv.destroyAllWindows()

4. Anisotropic diffusion filter

Perona-Malik anisotropic diffusion smooths flat regions but not across edges. Each direction's
contribution is weighted by a conduction coefficient that shrinks wherever the gradient is large
(i.e. at an edge): $c = e^{-(\nabla I / \kappa)^2}$. The image is then updated iteratively,
$I \mathrel{+}= \lambda \sum_{\text{directions}} c \cdot \nabla I$, so diffusion happens freely in
smooth areas and is blocked at edges.

In [ ]:
def anisotropic_diffusion(img, iterations=15, kappa=30, lam=0.15):

    img = img.astype(np.float32)

    for _ in range(iterations):

        north = np.roll(img, -1, axis=0) - img
        south = np.roll(img, 1, axis=0) - img
        east = np.roll(img, -1, axis=1) - img
        west = np.roll(img, 1, axis=1) - img

        c_n = np.exp(-(north / kappa) ** 2)
        c_s = np.exp(-(south / kappa) ** 2)
        c_e = np.exp(-(east / kappa) ** 2)
        c_w = np.exp(-(west / kappa) ** 2)

        img += lam * (c_n * north + c_s * south + c_e * east + c_w * west)

    return np.clip(img, 0, 255).astype(np.uint8)


diffused = anisotropic_diffusion(gaussian_img)

cv.imshow("Noisy", gaussian_img)
cv.imshow("Anisotropic Diffusion", diffused)

cv.waitKey(0)
cv.destroyAllWindows()

5. Fourier transform-based image denoising

Structured noise (e.g. the periodic noise above) concentrates its energy at specific frequencies,
which are often separable from the image's own frequency content. Transforming to the frequency
domain and multiplying by a low-pass mask $H = e^{-D^2/2D_0^2}$ suppresses that noise; the inverse
FFT then returns the cleaned spatial-domain image.

In [ ]:
height, width = img.shape

u, v = np.meshgrid(np.arange(width) - width // 2, np.arange(height) - height // 2)
D = np.sqrt(u**2 + v**2)

d0 = 40
mask = np.exp(-(D**2) / (2 * d0**2))

F = np.fft.fftshift(np.fft.fft2(periodic_img))
fft_denoised = np.abs(np.fft.ifft2(np.fft.ifftshift(F * mask)))
fft_denoised = np.clip(fft_denoised, 0, 255).astype(np.uint8)

cv.imshow("Noisy", periodic_img)
cv.imshow("FFT Denoised", fft_denoised)

cv.waitKey(0)
cv.destroyAllWindows()

6. Image Restoration
a. Point spread function
   i. Out-of-focus blur
   ii. Atmospheric turbulence blur
   iii. Motion blur

A degradation is modeled as convolution with a point spread function (PSF), $g = f * h$:
- Out-of-focus blur: a uniform disk PSF of a given radius.
- Motion blur: a line PSF along the direction of motion (spreads each point along that line).
- Atmospheric turbulence blur: a smooth, low-pass-like frequency response,
  $H = e^{-k D^{5/3}}$, that widens as turbulence increases.

In [ ]:
def out_of_focus_psf(shape, radius):

    psf = np.zeros(shape, dtype=np.float32)
    cy, cx = shape[0] // 2, shape[1] // 2

    y, x = np.ogrid[:shape[0], :shape[1]]
    psf[(x - cx) ** 2 + (y - cy) ** 2 <= radius ** 2] = 1

    return psf / psf.sum()


out_of_focus_kernel = out_of_focus_psf((15, 15), 5)
out_of_focus_blur = cv.filter2D(img, -1, out_of_focus_kernel)

# Motion blur: a straight line kernel smears the image along that line
motion_kernel = np.zeros((15, 15))
motion_kernel[7, :] = 1
motion_kernel /= motion_kernel.sum()

motion_blur = cv.filter2D(img, -1, motion_kernel)

# Atmospheric turbulence blur (frequency domain)
H_turbulence = np.exp(-0.0025 * (D ** (5 / 3)))
turbulence_blur = np.abs(np.fft.ifft2(np.fft.ifftshift(F * H_turbulence)))
turbulence_blur = np.clip(turbulence_blur, 0, 255).astype(np.uint8)

cv.imshow("Original", img)
cv.imshow("Out-of-Focus Blur", out_of_focus_blur)
cv.imshow("Atmospheric Turbulence Blur", turbulence_blur)
cv.imshow("Motion Blur", motion_blur)

cv.waitKey(0)
cv.destroyAllWindows()

b. Inverse filter
c. Pseudo-inverse filter
d. Weiner filter

Given the degradation model $G = F \cdot H$ (plus noise $N$ in practice):
- Inverse filter: recovers $\hat F = G / H$ directly - exact when there is no noise, but blows up
  wherever $H \approx 0$, amplifying any noise present.
- Pseudo-inverse filter: only inverts where $|H|$ exceeds a small threshold, setting the rest to
  zero, which avoids dividing by near-zero values.
- Wiener filter: balances restoration against noise using
  $\hat F = \dfrac{H^*}{|H|^2 + K} G$, where $K$ approximates the noise-to-signal ratio; small $K$
  behaves like the inverse filter, larger $K$ suppresses noise amplification at the cost of some
  remaining blur.

In [ ]:
d0_blur = 15
H = np.exp(-(D**2) / (2 * d0_blur**2))

# Degraded image = blur + noise
blurred = np.abs(np.fft.ifft2(np.fft.ifftshift(F * H)))
noise = np.random.normal(0, 5, img.shape)
degraded = np.clip(blurred + noise, 0, 255).astype(np.uint8)

G = np.fft.fftshift(np.fft.fft2(degraded))

# Inverse filter: divide out the degradation directly
inverse_restored = np.abs(np.fft.ifft2(np.fft.ifftshift(G / H)))

# Pseudo-inverse filter: ignore frequencies where H is nearly zero
H_inv = np.where(np.abs(H) > 0.05, 1 / H, 0)
pseudo_restored = np.abs(np.fft.ifft2(np.fft.ifftshift(G * H_inv)))

# Wiener filter: balances inversion against the noise-to-signal ratio K
K = 0.01
H_wiener = np.conj(H) / (np.abs(H) ** 2 + K)
wiener_restored = np.abs(np.fft.ifft2(np.fft.ifftshift(G * H_wiener)))

cv.imshow("Original", img)
cv.imshow("Degraded (Blur + Noise)", degraded)
cv.imshow("Inverse Filter", np.clip(inverse_restored, 0, 255).astype(np.uint8))
cv.imshow("Pseudo-Inverse Filter", np.clip(pseudo_restored, 0, 255).astype(np.uint8))
cv.imshow("Wiener Filter", np.clip(wiener_restored, 0, 255).astype(np.uint8))

cv.waitKey(0)
cv.destroyAllWindows()